# politbert - uk dataset
fine-tuning politbert on parlvote+
3-class: left / center / right
class weights needed because center is only ~7% of the data

In [ ]:
# install packages
! pip install transformers scikit-learn

In [ ]:
# imports 
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
print('imports done')

In [ ]:
# use gpu if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# file paths for uk data
train_path= '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/uk_train.csv'

val_path = '/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/uk_val.csv'

test_path='/kaggle/input/datasets/zainabrizvi5/thesis-data/final_dataset/uk_test.csv'
output_dir = '/kaggle/working/politbert_uk'

In [ ]:
model_name = 'maurice/PolitBERT'
max_length=256
batch_size=16
epochs=3
lr=2e-5

In [ ]:
# 3 classes for uk: left center right
label2id={'left': 0, 'center': 1, 'right': 2}
id2label={0: 'left', 1: 'center', 2: 'right'}

In [ ]:
# load data and check class distribution

train_df=pd.read_csv(train_path)
val_df=pd.read_csv(val_path)
test_df= pd.read_csv(test_path)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
# center class is only 7% therefore we need weights to prevent the model from ignoring it
train_labels=[label2id[l] for l in train_df['label'].tolist()]
class_weights= compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=np.array(train_labels))
class_weights_tensor= torch.tensor(class_weights, dtype=torch.float).to(device)


In [ ]:
# dataset class
class SpeechDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, label2id):
        self.texts= df['text'].tolist()
        self.labels= [label2id[l] for l in df['label'].tolist()]
        self.tokenizer =tokenizer
        self.max_length= max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length, padding='max_length',truncation=True,return_tensors='pt')
        
        return {
            'input_ids':enc['input_ids'].squeeze(),'attention_mask': enc['attention_mask'].squeeze(),'label':torch.tensor(self.labels[idx])}

In [ ]:
# load tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

In [ ]:
# create dataset split
train_dataset = SpeechDataset(train_df, tokenizer, max_length, label2id)
val_dataset = SpeechDataset(val_df,tokenizer, max_length, label2id)
test_dataset= SpeechDataset(test_df,tokenizer, max_length, label2id)

In [ ]:
# wrap in dataloaders
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader =DataLoader(test_dataset,batch_size=batch_size)

In [ ]:
# load model with 3 output labels
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)
model.to(device)
print('model loaded')

In [ ]:
# use weighted loss to handle the imbalanced center class
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer=AdamW(model.parameters(), lr=lr)
total_steps= len(train_loader) * epochs
scheduler=get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:

# training loop using weighted loss

def evaluate(model,loader, split='val'):
    model.eval()
    all_preds,all_labels= [], []


    with torch.no_grad():

        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask= batch['attention_mask'].to(device)
            labs =batch['label'].to(device)
            out  =model(input_ids=ids, attention_mask=mask)
            preds= torch.argmax(out.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())

    acc =accuracy_score(all_labels, all_preds)

    f1  =f1_score(all_labels, all_preds,average='macro')
    
    #print validation metrics
    print(f'{split} — acc: {acc:.4f}, macro f1: {f1:.4f}')
    return acc,f1

In [ ]:
# print history


history = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device)
        mask =batch['attention_mask'].to(device)
        labs=batch['label'].to(device)
        # using custom loss with class weights instead of model's built-in loss
        out  =model(input_ids=ids, attention_mask=mask)
        loss=loss_fn(out.logits, labs)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss=total_loss / len(train_loader)
    val_acc,val_f1 =evaluate(model, val_loader,split=f'epoch {epoch+1} val')

    history.append({'epoch': epoch+1, 'train_loss': avg_loss, 'val_acc': val_acc, 'val_f1': val_f1})
    print(f'epoch {epoch+1} train loss: {avg_loss:.4f}')

In [ ]:
# test evaluation
print('training done')
for h in history:
    print(h)

In [ ]:
# save 
print('final test evaluation')
test_acc, test_f1 = evaluate(model, test_loader, split='test')

In [ ]:
pd.DataFrame(history).to_csv(os.path.join(output_dir, 'training_history.csv'), index=False)
pd.DataFrame([{'model': 'politbert', 'dataset': 'UK (ParlVote+)', 'test_acc': test_acc, 'test_f1': test_f1,'epochs': epochs, 'batch_size': batch_size, 'lr': lr, 'max_length': max_length, 'class_weights': list(class_weights)}]).to_csv(os.path.join(output_dir, 'final_results.csv'), index=False)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)